In [ ]:
# libraries

from io import BytesIO
import boto3
import pandas as pd
import requests
from botocore.config import Config

In [ ]:
"""
FUNCTION
"""

def ingest_population_data(
    bucket_name,
    aws_region,
    s3_prefix="raw/enrichment/population"
):
    """
    Download the ONS population workbook, convert the required
    worksheet to CSV, and upload the CSV to S3.

    The original Excel workbook is already retained separately
    in the raw S3 population location.

    Returns
    -------
    str
        Full S3 location of the uploaded CSV.
    """

    population_url = (
        "https://www.ons.gov.uk/file?uri="
        "%2Fpeoplepopulationandcommunity"
        "%2Fpopulationandmigration"
        "%2Fpopulationestimates"
        "%2Fadhocs"
        "%2F3194populationestimatesforpoliceforceareasinenglandandwales"
        "bysingleyearofageandsexmid1991tomid2024"
        "%2Fpoliceforceareas1991to2024.xlsx"
    )

    # Download the Excel workbook
    print("Downloading population data from ONS...")

    response = requests.get(
        population_url,
        timeout=120
    )

    response.raise_for_status()

    print(f"Downloaded {len(response.content):,} bytes.")

    # Read the required worksheet
    print("Reading worksheet: Mid-2021 to Mid-2024...")

    population_df = pd.read_excel(
        BytesIO(response.content),
        sheet_name="Mid-2021 to Mid-2024",
        header=None,
        engine="openpyxl"
    )

    print(
        f"Worksheet read: {population_df.shape[0]:,} rows, "
        f"{population_df.shape[1]:,} columns."
    )

    # Convert the worksheet to CSV bytes
    csv_content = population_df.to_csv(
        index=False,
        header=False
    ).encode("utf-8")

    # Create an S3 client with automatic retry settings
    s3 = boto3.client(
        "s3",
        region_name=aws_region,
        config=Config(
            retries={
                "max_attempts": 10,
                "mode": "standard"
            }
        )
    )

    # Destination for the CSV
    csv_s3_key = (
        f"{s3_prefix}/csv/"
        "ons_police_force_population_2021_2024.csv"
    )

    print("Uploading CSV to S3...")

    # Upload the CSV from memory
    s3.upload_fileobj(
        Fileobj=BytesIO(csv_content),
        Bucket=bucket_name,
        Key=csv_s3_key,
        ExtraArgs={
            "ContentType": "text/csv"
        }
    )

    csv_s3_location = (
        f"s3://{bucket_name}/{csv_s3_key}"
    )

    # Confirm that the uploaded object exists
    uploaded_object = s3.head_object(
        Bucket=bucket_name,
        Key=csv_s3_key
    )

    print(f"Uploaded: {csv_s3_location}")
    print(f"Uploaded size: {uploaded_object['ContentLength']:,} bytes.")
    print("Population ingestion complete.")

    return csv_s3_location

In [ ]:
"""
CALL
"""

population_csv_s3_location = ingest_population_data(
    bucket_name="rockborne-ch19-g1-crime",
    aws_region="us-west-2"
)

print(population_csv_s3_location)